In [0]:
%pip install dbldatagen
dbutils.library.restartPython()

In [0]:
import dbldatagen as dg
from pyspark.sql.types import StringType, IntegerType, DecimalType, DateType

# Disable ANSI mode to avoid divide-by-zero errors
spark.conf.set("spark.sql.ansi.enabled", "false")

# Create schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

# ---------------------------
# 1. CUSTOMERS
# ---------------------------
customers_spec = (
    dg.DataGenerator(spark, name="customers", rows=50000, partitions=4)
    .withIdOutput()
    .withColumn("first_name", "string", template=r"\\w \\w", random=True)
    .withColumn("signup_date", "date", begin="2020-01-01", end="2026-07-01", random=True)
    .withColumn("region", "string", random=True,
                values=["North", "South", "East", "West", "Central"],
                weights=[0.25, 0.20, 0.20, 0.20, 0.15])
    .withColumn("customer_segment", "string", random=True,
                values=["Regular", "Premium", "VIP"],
                weights=[0.70, 0.25, 0.05])
)
df_customers = customers_spec.build().withColumnRenamed("id", "customer_id")
#df_customers.display()
#df_customers.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.customers")
df_customers.write.mode("overwrite").json("/Volumes/workspace/default/raw_landing/customers/")

# ---------------------------
# 2. PRODUCTS
# ---------------------------
products_spec = (
    dg.DataGenerator(spark, name="products", rows=2000, partitions=2)
    .withColumn("product_id", "long", minValue=1, maxValue=2000, uniqueValues=2000)
    .withColumn("category", "string",random=True,
                values=["Electronics", "Apparel", "Home", "Beauty", "Sports", "Books"],
                weights=[0.25, 0.25, 0.15, 0.15, 0.10, 0.10])
    .withColumn("unit_price", "decimal(10,2)", minValue=5.00, maxValue=800.00, random=True)
    .withColumn("cost_price", "decimal(10,2)", expr="unit_price * 0.60")
)
df_products = products_spec.build()
#df_products.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.products")
df_products.write.mode("overwrite").json("/Volumes/workspace/default/raw_landing/products/")

# ---------------------------
# 3. ORDERS (skewed toward fewer customers, seasonal spikes)
# ---------------------------
orders_spec = (
    dg.DataGenerator(spark, name="orders", rows=1_000_000, partitions=8)
    .withColumn("order_id", "long", minValue=1, maxValue=1_000_000, uniqueValues=1_000_000)
    # Zipf-like skew: a small % of customers generate most orders
    .withColumn("customer_id", "long", minValue=1, maxValue=50000, distribution="normal", random=True)
    .withColumn("product_id", "long", minValue=1, maxValue=2000, random=True)
    .withColumn("quantity", "int", minValue=1, maxValue=5, distribution="normal", random=True)
    .withColumn("order_date", "date", begin="2024-01-01", end="2026-07-17", random=True)
    .withColumn("order_status", "string",random=True,
                values=["Completed", "Cancelled", "Returned", "Pending"],
                weights=[0.80, 0.08, 0.07, 0.05])
    .withColumn("payment_method", "string",random=True,
                values=["Card", "PayPal", "MobilePay", "COD"],
                weights=[0.55, 0.20, 0.20, 0.05])
)
df_orders = orders_spec.build()
#df_orders.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.orders")
df_orders.write.mode("overwrite").json("/Volumes/workspace/default/raw_landing/orders/")

print("Rows generated -> customers:", df_customers.count(),
      "| products:", df_products.count(),
      "| orders:", df_orders.count())